In [1]:
%run 10_MNESIS_polychronous-chains.ipynb

datetag = '2026-04-21'
Files in that folder
total 1240
-rw-r--r--@  1 laurentperrinet  staff    5048 22 avr.  11:48 01_MNESIS_boilerplate.ipynb
-rw-r--r--   1 laurentperrinet  staff    8487 22 avr.  12:39 25_MNESIS_optuna.ipynb
-rw-r--r--   1 laurentperrinet  staff   21285 22 avr.  12:39 10_MNESIS_polychronous-chains.ipynb
-rw-r--r--   1 laurentperrinet  staff  169696 22 avr.  12:40 11_MNESIS_learn-synthetic.ipynb
-rw-r--r--@  1 laurentperrinet  staff    4399 22 avr.  12:50 05_MNESIS_parameters.ipynb
-rw-r--r--   1 laurentperrinet  staff    7582 22 avr.  12:52 16_MNESIS_testing-trigger-fraction.ipynb
-rw-r--r--@  1 laurentperrinet  staff    7933 22 avr.  12:54 20_MNESIS_scanning-parameters.ipynb
-rw-r--r--@  1 laurentperrinet  staff  120255 22 avr.  12:57 13_MNESIS_testing-inference.ipynb
-rw-r--r--   1 laurentperrinet  staff  120088 22 avr.  12:58 15_MNESIS_testing-trigger-duration.ipynb
-rw-r--r--   1 laurentperrinet  staff  121628 22 avr.  12:58 14_MNESIS_testing-noise.ipynb
drwxr-x

## learning Spiking Heidelberg Digits patterns

In [ ]:
opt = Params()
hd = HD_SNN(opt)
hd.net.to(hd.opt.device)

# Make the target periodic
hd.target[:, :, (-hd.opt.num_delay):] = hd.target[:, :, :hd.opt.num_delay]

model_filename = data_cache / f"{hd.opt.datetag}_shd.pth"
lock_filename = data_cache / model_filename.with_suffix('.lock')
# if False:
if RECOMPUTE:
    model_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
    lock_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
try:
    model_state_dict = torch.load(model_filename, map_location=torch.device(hd.opt.device))
    hd.net.load_state_dict(model_state_dict)
    hd.net.eval()
    lock_filename.unlink(missing_ok=True) # in case the lock file was not unlinked
    print(f"Model weights loaded from {model_filename}") # Add a log message
except FileNotFoundError:
    if not lock_filename.exists():
        print(f"Model file not found: {model_filename}, intitializing the new model.")
        lock_filename.touch(exist_ok=True)
        ##################
        hd.update_weight()
        hd.learn_model()
        ##################
        torch.save(hd.net.state_dict(), model_filename)
        lock_filename.unlink(missing_ok=True)
    else:
        print(f"Model file is locked: {lock_filename}, passing.")

Model file not found: ../cached_data/2026-04-21_periodic.pth, intitializing the new model.


In [ ]:
with torch.no_grad():
    target_full = torch.nn.functional.pad(hd.target, (opt.N_pretime, opt.N_pretime))
    input_spikes = hd.get_input_spikes(p_A=hd.opt.p_A, N_pretime=hd.opt.N_pretime, N_trigger_time=hd.opt.num_delay)
    _, _, spikes = hd.forward_pass(input_spikes)
    spikes_evoked = spikes[:, :, (hd.opt.N_pretime+hd.opt.num_delay):(-hd.opt.N_pretime)]
    target_evoked = hd.target[:, :, hd.opt.num_delay:]

    precision, recall, f1_score = get_scores(spikes_evoked, target_evoked)
    print(f'precision = {precision:.3f}\t recall = {recall:.3f}\t f1_score = {f1_score:.3f} ')

In [ ]:
fig,ax = plt.subplots(figsize=(13, 8))
splt.raster(spikes[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, c="blue", marker='+', alpha=.5)
splt.raster(target_full[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, facecolor='none', edgecolor='red',  marker='o', alpha=.5)
splt.raster(input_spikes[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, c="green", marker='x', alpha=.5)
ax.vlines([opt.N_pretime], 0, opt.N_neuron_show, 'r', ls='--')
ax.vlines([opt.N_pretime+opt.num_delay], 0, opt.N_neuron_show, 'b', ls='--')
ax.set_xlabel("Time step (ms)")
ax.set_ylabel("Neuron Address")
ax.set_ylim(-1., opt.N_neuron_show + 1.)
if figpath is not None: printfig(fig, 'target', fig_width=opt.fig_width, fig_height=opt.fig_height)
spikes.shape, spikes[i_SM, :, :].sum().item(), hd.target[i_SM, :, :].sum().item()

In [ ]:
# hd.net.lin.bias.cpu().detach().numpy()
fig,ax = plt.subplots(figsize=(13, 8))
ax.hist(hd.net.lin.weight.cpu().detach().numpy().ravel(), bins=100, density=True) # pyright: ignore[reportCallIssue, reportAttributeAccessIssue]
ax.set_yscale('log')